# Lab 04 — ML Anomaly Detection

使用 sliding-視窗 features and Isolation Forest 進行多變量異常偵測，並用視覺化比較事件敏感度。

## Course architecture boundary

本課程有兩條資料路徑，請不要混在一起：

- **Actual operating path**: OS / network signals -> exporter -> Prometheus -> Grafana.
- **Python learning path**: organized telemetry CSV or previous notebook output -> Python analysis -> result CSV -> `outputs/prometheus-dropzone/current_results.csv` -> `python_results_exporter` -> Prometheus -> Grafana.

除非 cell 明確寫的是 operational deployment example，Python anomaly / forecasting / RCA notebooks 的主要輸入是 organized telemetry CSV 或前一個 notebook 的輸出，不是 Prometheus query。Grafana 也不直接讀 CSV；Grafana 查 Prometheus。


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab04_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab04_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab04_pipeline_position.svg")


## Lab 04 概覽

### 學習目標

1. 理解「多變量」異常偵測與單變量方法的本質差異
2. 掌握 Isolation Forest 的隨機隔離原理與 contamination 參數的業務意義
3. 用 feature contribution 解釋 ML 模型的異常分數
4. 比較 Isolation Forest 與其他 ML 方法的適用場景

### 前置條件

Lab 01 完成（`outputs/self-study/features.csv` 存在）

## 設計主線：ML 應該補上單變量方法看不到的盲點

本章的實務連結是複合異常。單一 metric 可能都沒有超過 threshold，但多個 metric 同時偏離時，事件已經有明確風險。

設計 ML 偵測時請問三個問題：

1. **是否真的需要 ML**：如果 Z-score 或 SPC 已經能解釋問題，ML 可能只增加維護成本。
2. **特徵組合是否有實務語意**：Isolation Forest 分數要能回頭說明是哪幾個 feature 推動異常。
3. **部署位置是否合適**：較重的 ML scoring 通常不適合直接塞進 Prometheus rule，應考慮獨立 scoring service。

設計原則：ML 的價值不是「更高級」，而是能抓到規則方法看不到的組合型態。


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5 分鐘，請先不要執行 cell。

---

## 📖 概念：什麼是多變量異常偵測？

### 單變量 vs 多變量的根本差異

Lab 02 和 Lab 03 的方法是**單變量**的：每次只分析一個 metric（`error_rate`、`traffic_total`...）。
這有一個盲點：某些異常只在**多個 metric 的組合**中才看得出來。

**反例**：Broadcast storm 事件

| Metric | 單獨看 | 組合看 |
|---|---|---|
| `traffic_total` | 小幅上升（z = 1.8，未觸發）| |
| `broadcast_total` | 中幅上升（z = 2.5，未觸發）| |
| `error_rate` | 輕微上升（z = 1.5，未觸發）| |
| 三個一起 | | **異常！** 三個維度同時偏離，這個組合極不尋常 |

多變量方法的核心問題：「在所有特徵的**聯合分布**下，這個時間點的組合有多不尋常？」

### Feature Matrix 的概念

把每個時間窗口想像成多維空間中的一個點：
- 一個時間點有 N 個特徵 → 一個 N 維空間中的點
- 「正常」的時間點聚集在空間中某些區域
- 「異常」的時間點落在稀疏的外圍

多變量異常偵測的任務：找出落在外圍的點。

## Step 1 - 建立多變量 視窗 features

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

In [ ]:
model_features = [
    "traffic_total", "ucast_total", "avg_packet_size", "error_rate",
    "discard_rate", "broadcast_ratio", "multicast_ratio", "unknown_total",
]

window_rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").set_index("timestamp")
    wf = g[["device_id", "port_id", "port_role", "event_label", "event_id"]].copy()
    generated = {}
    for col in model_features:
        roll = g[col].rolling("30min", min_periods=3)
        generated[f"{col}_mean_30m"] = roll.mean()
        generated[f"{col}_std_30m"] = roll.std()
        generated[f"{col}_max_30m"] = roll.max()
        generated[f"{col}_min_30m"] = roll.min()
        generated[f"{col}_p95_30m"] = roll.quantile(0.95)
        generated[f"{col}_slope_30m"] = g[col].diff(6)
    wf = pd.concat([wf, pd.DataFrame(generated, index=wf.index)], axis=1)
    window_rows.append(wf.reset_index())

windows = pd.concat(window_rows, ignore_index=True).fillna(0)
X_cols = [c for c in windows.columns if c.endswith(("mean_30m", "std_30m", "max_30m", "min_30m", "p95_30m", "slope_30m"))]
X = windows[X_cols].replace([np.inf, -np.inf], 0).fillna(0)
print(windows.shape, len(X_cols))
display(windows[["timestamp", "port_id", "event_label"] + X_cols[:6]].head())

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 8 分鐘，請先不要執行 cell。

---

## 📖 概念：Isolation Forest 原理

### 核心思想：異常點更容易被「隔離」

Isolation Forest 的直觀來自一個簡單觀察：
**異常點通常是孤立的，正常點通常是聚集的。**

試想把資料放在一個多維空間，然後隨機劃線（用隨機特徵、隨機切割點）。
- 正常的點有很多鄰居——你需要切很多刀才能把它和所有鄰居分開
- 異常的點是孤立的——切一兩刀就把它隔離了

**Isolation Tree 的建立方式**：
1. 隨機選一個特徵
2. 在該特徵的最小值和最大值之間，隨機選一個切割值
3. 切割後遞迴地對左右子集重複
4. 直到每個資料點都被獨立（或達到樹的最大深度）

**異常分數 = 平均隔離深度的倒數**：
一個點平均需要多少步才能被隔離？步數少 → 異常分數高 → 更可能是異常。

```
          few splits               many splits
          (2–3 cuts)              (8–12 cuts)
               ●  ← anomaly           ●●●●●●
                                    ●●●●●●  ← normal cluster
                                    ●●●●●●
```

### `contamination` 參數的業務含義

```python
IsolationForest(contamination=0.05)
```

`contamination` 告訴模型：「你認為資料中大約有多少比例是異常的？」

- `0.05` = 預期有 5% 的點是異常的
- 模型據此設定 anomaly score 的切點
- **這不是統計輸出——這是你的業務假設**

在正常運作的網路中，異常率遠低於 5%。若你把 contamination 設太高，
模型會把正常的流量高峰也標記為異常。

**如何選擇 contamination？**

若你有歷史事件記錄，可以估算：
```
contamination ≈ total anomaly time points / total time points
```
若沒有，從 `0.01`（1%）開始，觀察標記的點是否符合你的直覺。

### 為什麼選 Isolation Forest？

1. 不假設資料分布（無參數方法）——網路流量不是常態分布
2. 計算效率高（O(n log n)）——適合大量時間序列
3. 結果可以用 feature contribution 部分解釋（Lab 05 會示範）
4. 相對穩健——對高維資料（多個 metric）的表現優於 LOF 等方法

## Step 2 - 訓練 Isolation Forest

非監督式模型不需要 anomaly label，但 contamination 會影響異常比例。

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(n_estimators=250, contamination=0.035, random_state=42)
model.fit(X_scaled)
windows["ml_anomaly_score"] = -model.score_samples(X_scaled)
windows["ml_is_anomaly"] = model.predict(X_scaled) == -1
windows["ml_method"] = "IsolationForest"

display(windows.loc[windows["ml_is_anomaly"], ["timestamp", "port_id", "event_label", "ml_anomaly_score"]].head(12))

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 - 視覺化：anomaly score trend

In [ ]:
port_id = "port-id7428"
p = windows[windows["port_id"] == port_id].copy()
threshold = windows["ml_anomaly_score"].quantile(0.965)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p["ml_anomaly_score"], label="ML anomaly score", linewidth=1)
ax.axhline(threshold, color="tab:red", linestyle="--", label="global 96.5% threshold")
v = p[p["ml_is_anomaly"]]
ax.scatter(v["timestamp"], v["ml_anomaly_score"], color="tab:red", s=18, label="ML anomaly")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Isolation Forest anomaly score")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 - 視覺化：Top anomalous 視窗s 與事件敏感度

In [ ]:
top_windows = windows.sort_values("ml_anomaly_score", ascending=False).head(15)
display(top_windows[["timestamp", "port_id", "event_label", "event_id", "ml_anomaly_score"]])

event_score = (
    windows.groupby("event_label")
    .agg(avg_score=("ml_anomaly_score", "mean"), p95_score=("ml_anomaly_score", lambda s: s.quantile(0.95)), anomaly_rate=("ml_is_anomaly", "mean"), rows=("timestamp", "size"))
    .sort_values("p95_score", ascending=False)
)
display(event_score)

fig, ax = plt.subplots(figsize=(12, 5))
event_score["anomaly_rate"].sort_values().plot(kind="barh", ax=ax, color="tab:blue")
ax.set_title("ML anomaly rate by simulated event")
ax.set_xlabel("anomaly rate")
plt.tight_layout()
show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 - 簡易 feature contribution

Isolation Forest 本身不是可解釋模型，這裡用 top anomalous 視窗s 與全體 median 的 robust distance 作為教學用 contribution proxy。

In [ ]:
top = windows[windows["ml_is_anomaly"]].copy()
median = X.median()
mad = (X - median).abs().median().replace(0, 1)
contrib = ((top[X_cols] - median).abs() / mad).mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
contrib.sort_values().plot(kind="barh", ax=ax, color="tab:purple")
ax.set_title("Proxy feature contribution among anomalous windows")
ax.set_xlabel("mean robust distance")
plt.tight_layout()
show_fig(fig)

In [ ]:
output_cols = ["timestamp", "device_id", "port_id", "port_role", "event_label", "event_id", "ml_method", "ml_anomaly_score", "ml_is_anomaly"] + X_cols
out_path = DATA_PROCESSED / "ml_anomaly_scores.csv"
windows[output_cols].to_csv(out_path, index=False)
print(f"saved: {out_path}")

---

## ⚖️ 方法比較：Isolation Forest vs DBSCAN vs Autoencoder

| | Isolation Forest | DBSCAN | Autoencoder（深度學習） |
|---|---|---|---|
| **原理** | 隨機隔離深度 | 密度聚類，孤立點為異常 | 重建誤差：重建失敗的點為異常 |
| **訓練資料需求** | 無監督，資料量中等 | 無監督，資料量中等 | 需大量正常資料；訓練時間長 |
| **可解釋性** | 中（feature contribution） | 低 | 低（黑盒） |
| **高維資料** | 好（設計來處理高維） | 差（維度詛咒嚴重） | 好（但需要更多資料） |
| **計算成本** | 低 | 中 | 高（需要 GPU 或大量Time） |
| **適合場景** | 即時偵測、資料量有限 | 空間聚類明顯的資料 | 資料量大、追求最高準確度 |

### 本課程選 Isolation Forest 的原因

1. 網路監控的特徵數量適中（10–20 個），Isolation Forest 在這個範圍表現穩定
2. 無須假設資料分布，適合真實網路流量的長尾特性
3. 訓練快、可在 CPU 上即時重訓練（當網路特性改變時）
4. 可以計算 feature contribution，保留可解釋性（告警上要給工程師看「為什麼」）

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：Contamination 的影響

在 Step 2 的 code cell 中找到 `contamination=0.05`，改成 `0.01`，重新執行 Step 3 和 Step 4。

觀察：
- 標記為異常的時間點數量是否下降？
- 已知事件期間的 anomaly score 是否仍然高於其他時段？
- `contamination = 0.01` 時，哪些事件仍然被偵測到，哪些被錯過了？

**判斷標準**：好的 contamination 設定應讓已知事件的 anomaly score 明顯高於正常時段，
同時在「正常的忙碌時段」（非事件）的 score 保持在 閾值 以下。

---

> 「Isolation Forest 的 random seed 如果改變，結果會有多大差異？這對生產部署有什麼影響？」

> 「feature contribution 顯示 `broadcast_total` 是最重要的特徵——這是演算法「理解」了廣播風暴嗎？還是別的原因？」

> 「如果你的網路每季度有一次大型備份（流量暴增），Isolation Forest 的訓練資料應該包含還是排除這個備份時段？」

## ⚠ 人類驗證點 #4 — contamination 是你對業務的假設，不是模型的答案

Isolation Forest 的 `contamination` 參數是你告訴模型「大概有多少比例的資料是異常的」。
這個數字來自業務知識，不是統計計算。

### 如何判斷

`contamination=0.05` 的意思是：「我預期這份資料裡約有 5% 是異常的。」

問自己：
- 你的合成資料裡事件覆蓋了幾個百分比的時間點？（參考 events.csv）
- 在真實環境，你預期一個月裡有幾天是有問題的？
- 把這個比例代入 contamination，是否抓到了你已知的事件？

### 讓你重新考慮的信號

- `contamination=0.05` 但模型沒有標到任何已知事件 → 可能太低，或特徵設計有問題
- `contamination=0.2` 且正常時段充滿紅點 → 太高，代表你告訴模型「每 5 個點就有 1 個異常」

### 🔧 探索練習

在 Step 2 的 code cell 找到以下參數，修改並重新執行 Step 3 的視覺化：

```python
contamination = 0.05   # change to 0.02 and 0.10, observe how the anomaly score distribution shifts
n_estimators = 100     # change to 50, observe the speed vs. result-stability tradeoff
```

> 「contamination 調高，精確率（precision）和召回率（recall）哪個會上升？為什麼？」

> 「Isolation Forest 的 feature contribution 告訴我們哪個特徵最重要。這和 Lab 02 的單變量結果一致嗎？如果不一致，說明了什麼？」

> 「這個模型能不能部署成即時系統？需要做什麼？」（引導到 scoring service sidecar 概念）


## Optional: update Grafana with this result

Notebook 會顯示 ML anomaly score 與 anomaly flag。若想在 Grafana 看同一個模型輸出，保持 `python infra/python_results_exporter.py` 執行，然後複製：

```bash
cp outputs/self-study/ml_anomaly_scores.csv outputs/prometheus-dropzone/current_results.csv
```

建議 PromQL：`aiops_python_result{column="ml_anomaly_score"}`、`aiops_python_result{column="ml_is_anomaly"}`。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。
